# 14 - The Governed Agent, End To End

Every other notebook proves one decision at a time. This one gives a
LangGraph agent a realistic multi-part brief and lets governance
visibly change its course:

> *"Build the EU churn dashboard, push the at-risk segment to
> Salesforce, and fine-tune the support assistant on ticket text."*

The agent reads the rulebook first, self-revises warned SQL, turns a
conditional export into a human exception packet and resumes with
attested controls, REROUTES a denied fine-tune to the governed
alternative, and closes by chaining `explain_why` over every decision
id it collected. Offline it replays recorded answers with a
deterministic planner; in live mode an optional LLM drafts the SQL
(`METATATE_EXAMPLES_LLM`) while governance calls stay identical.

Requires `requirements-framework.txt` (LangGraph), like notebook 11.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the AcmeCloud demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


def asset(table, column=None, schema="public"):
    ref = {"database": "acmecloud_demo", "schema": schema, "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


## Run the whole arc


In [ ]:
from governed_agent_arc import ArcRecordingClient, ScriptedPlanner, run_arc

recording = ArcRecordingClient(client)
report = run_arc(recording, ScriptedPlanner())
for line in report.transcript:
    print(line)


## The decision spine

Eleven Metatate calls, in order — the agent's entire interaction with
governance, every one a typed answer:


In [ ]:
for step, call in enumerate(report.decision_sequence, start=1):
    print(f"{step:>2}. {call['tool']:<24} -> {call['label']}")


## Beat 1: the draft that had to change

The first draft referenced a masked column — `warn`, not a guess. The
agent revised once and re-validated to `pass`. It never returns SQL
Metatate has not passed.


In [ ]:
print(f"draft:   {report.draft_sql}")
print(f"final:   {report.final_sql}")
print(f"revisions: {report.revision_count} (dashboard {report.dashboard_status})")


## Beat 2: the conditional export became a review, then a resume

`conditional` is not a soft yes — the agent built an exception packet
(the same `human_exception_workflow` machinery from notebook 09) and
resumed only after the reviewer attested BOTH required controls.


In [ ]:
packet = report.exception_packet
print(f"packet:  {packet['packet_id']} -> queue {packet['reviewer_queue']}")
print(f"attestations required: {packet['required_attestations']}")
print(f"status:  {report.export_status}")
print(f"resume:  {report.resume_payload['action']}")


## Beat 3: deny became redirection, not a dead end

Fine-tuning on raw ticket text is denied. The rulebook already showed
where training IS allowed — the agent re-asked on the feature store
and got a real `allow`, with its own decision id.


In [ ]:
print(f"training: {report.training_status} (rerouted: {report.rerouted})")


## Beat 4: the receipts

Every decision the agent collected resolves through `explain_why`,
and every one is still current in the live publication.


In [ ]:
for explanation in report.explanations:
    print(f"{explanation['decision_id']} -> current: {explanation['current']}")
print(f"total Metatate calls: {report.metatate_calls}")
